In [1]:
import csv
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score



# load the dataset

# In this step we load the CSV file that contains movie
# overviews and the corresponding labels (Drama or not Drama).

data = pd.read_csv(
    'data/Movie_Overview_Classification.csv',
    encoding='utf8',
    delimiter=',',
    quotechar='|',
    quoting=csv.QUOTE_MINIMAL
)



# preprocessing

# we remove missing values because the model cannot work
# with empty text or missing labels.

data = data.dropna(subset=['overview', 'genre_Drama'])

# we define our input (X) as the movie overview text
# and our output (y) as the target label.

X_text = data['overview']
y_genre = data['genre_Drama']



# baseline model

# First, we build a simple baseline model.
# We use TF-IDF to convert text into numerical features
# and Multinomial Naive Bayes as a basic classifier.

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

# We evaluate the model using 5-fold cross-validation
# to get a more reliable accuracy score.

baseline_scores = cross_val_score(
    baseline_model,
    X_text,
    y_genre,
    cv=5,
    scoring='accuracy'
)

print("Baseline Accuracy:", baseline_scores.mean())


# optimized model

# now we try to improve the model by enhancing preprocessing
# and using a stronger classifier.

# improvements we apply:
# - removing English stop words
# - using word pairs (n-grams)
# - filtering rare and very frequent words

optimized_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=3,
        max_df=0.85,
        sublinear_tf=False,
        binary=True,
        smooth_idf=True
    )),
    ("clf", MultinomialNB())
])
# again, we evaluate using 5-fold cross-validation

optimized_scores = cross_val_score(
    optimized_model,
    X_text,
    y_genre,
    cv=5,
    scoring='accuracy'
)

print("Optimized Accuracy:", optimized_scores.mean())



# comparison of results

# finally, we compare both models to see whether our
# improvements actually helped.

print("\n--- Final Comparison ---")
print("Baseline Accuracy:", baseline_scores.mean())
print("Optimized Accuracy:", optimized_scores.mean())
print("Difference:", optimized_scores.mean() - baseline_scores.mean())

Baseline Accuracy: 0.6647712743089095
Optimized Accuracy: 0.6768019162372069

--- Final Comparison ---
Baseline Accuracy: 0.6647712743089095
Optimized Accuracy: 0.6768019162372069
Difference: 0.012030641928297392
